# Let's Play CrewRift 🛰️

**Eight crew wake up on a drifting ship, and two of them are lying. Every player here is
a program.**

CrewRift is a social-deduction league where the players are bots: your policy walks the
corridors, does its tasks, watches who vanishes near a vent, casts its vote, and, when it
draws the imposter card, picks its moment to strike. This notebook is the hands-on tour.
It starts from zero, and by the end you will have:

1. installed the bundled **`swgy`** libraries,
2. built a working, role-aware **starter policy** for CrewRift, a step at a time,
3. **validated** it with an offline fuzz harness and seen how to **submit** it to the league,
4. toured the **navigation-mesh tooling** and **tuned** navigation knobs against an offline
   benchmark.

When your policy is playing ranked games, its replays become data: the companion notebook,
**`LetsAnalyzeCrewrift.ipynb`**, mines them for insight — task swimlanes, kill cams with
line-of-sight witnesses, heatmaps, and the movement-vs-optimal analyses that tell you what
*not* to spend time tuning.

Everything here runs from the wheels in `./libs/`. It needs no game engine and no local
checkouts. To submit your policy and pull your
own replays you'll want a free softmax account: sign up (or sign in) at
[softmax.com](https://softmax.com).

Building with a coding assistant? Point it at the contract cheat sheet in §2b and at the
`policy.py` file §3i writes; together they carry the contracts generated code most often gets
wrong.

---

## What is CrewRift?

**CrewRift** is a social-deduction game in the *Among Us* genre, played on a single fixed
map called **Croatoan**. Each round is a closed-roster **8-player** game with **2 imposters**:

| | Crewmates (6) | Imposters (2) |
|---|---|---|
| **Goal** | finish your assigned **tasks**, and vote the imposters out | **kill** crew until you reach parity (imposters ≥ crew), without being caught |
| **Do** | walk to a task station, hold **A**, stand still until it completes | blend in, fill your **kill bar**, strike an adjacent crewmate, then flee |
| **Move** | walk the corridors | walk, and also use **vents** to teleport/hide |
| **Meetings** | call an emergency meeting or **report** a body → everyone votes | fake tasks, misdirect, vote to survive |

When a body is reported or the emergency button is pressed, play pauses for a **vote**:
each player votes for a suspect or **skips**. Votes are final. Crew win by completing all
tasks **or** ejecting both imposters; imposters win by killing enough crew.

### Scoring (the training reward)

The engine emits a **dense per-action reward**. This is what your policy is optimising:

| Event | Points |
|---|---:|
| Win the game | **+100** |
| Complete a task | **+1** |
| Kill a crewmate | **+10** |
| **Not** voting *and* not skipping | **−10** |
| Idling (per 20s) | **−1** |

> Note that abstaining costs **−10**: skip is the safe floor for a policy with no read on
> the room. Our starter treats it that way, and its crew brain upgrades to backing a
> forming ejection once the live tally shows a clear leader (§3d).

*How does a bot even see this game?* No pixels — the server streams a labelled
scene graph, and `swgy_base` turns it into a clean `WorldState` every tick
(1 tick = 1/24 s). §2 shows exactly what your policy reads.


## 1 · Setup: install the bundled wheels

### The `swgy` toolkit

The toolkit ships as wheels in `./libs/`:

| Wheel | Import | What it gives you |
|---|---|---|
| **`swgy_base`** | `swgy_base` | the everything-you-need base layer: `protocol` (parse `sprite_v1`), `world` (the `WorldState` model), `nav` (walkability grid + nav-mesh + A\* + inertia-aware follower), and `runtime` (the websocket play loop + the `Policy` interface). **A policy depends only on this.** |
| **`swgy_tools`** | `swgy_tools` | offline dev tooling: capture streams, **build & visualise nav-meshes**, benchmark navigation. |
| **`swgy_tune`** | `swgy_tune` | a tiny bookkeeping layer for tuning: `Knob`/`KnobSpec` map bounded knobs to a genome vector and back (the knob-tuning section, §8, uses it to sample and decode `NavParams` candidates). |

Related projects you'll meet later: **[softmax-cli](https://github.com/Metta-AI/softmax-cli)**
(authentication), **[coworld](https://github.com/Metta-AI/coworld)** (the CLI that runs games, submits policies, and downloads
replays), **[coworld-crewrift](https://github.com/Metta-AI/coworld-crewrift)** (the game itself),
and more advanced reference policies to graduate to.

We install straight into **this notebook's own kernel** with `%pip`, so the packages are
importable in the cells below regardless of how you launched Jupyter.

`swgy_tools` depends on `swgy_base`, so we could install just `swgy_tools`, but we name each
wheel for clarity (`swgy_tune` supplies the knob-to-genome mapping used in §8). Installing pulls in `numpy` and
`websockets` from PyPI, so this cell needs the network on first run.

The cells build on each other: the smoke test below defines the `mesh` object that §3 and the
later tooling sections reuse. Use **Run All**, or run top to bottom.

In [ ]:
# needs network on first run (dependency resolution from PyPI)
#   swgy-tools brings numpy + matplotlib; nothing else is needed.
%pip install --quiet ./libs/swgy_base-0.3.0-py3-none-any.whl ./libs/swgy_tools-0.5.2-py3-none-any.whl ./libs/swgy_tune-0.0.1-py3-none-any.whl


In [ ]:
# Smoke test: import the pieces we'll use and load the bundled Croatoan nav-mesh.
import swgy_base
from swgy_base.runtime import Action, Policy      # the policy interface
from swgy_base.world import Phase, WorldState     # the world model
from swgy_base.nav import load_default_mesh       # the Croatoan mesh ships inside the wheel

mesh = load_default_mesh()
print("swgy_base", swgy_base.__version__, "ready")
print(f"Croatoan mesh: {len(mesh.nodes)} nodes, {len(mesh.edges)} edges, "
      f"{mesh.bounds[0]}x{mesh.bounds[1]} px map")

### Meet Croatoan

Every game plays out on this ship. The schematic below is generated entirely from the nav-mesh
you just loaded: the floor is the true walkable area, squares are task stations, diamonds are the
vents imposters teleport through, and the star is the one emergency button. Everything your
policy needs in order to navigate, hunt, or hide ships inside the `swgy_base` wheel.

**Run the next cell even if you skim the figure** — it defines the shared dark plot
style (`dark_ax`, `dark_legend`, the walk mask) that every later figure needs.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import ListedColormap

from swgy_base.world import BUTTON_RECT
from swgy_tools.plotstyle import EDGE, PAGE, SECONDARY, TASK_MARK, VENT_MARK, croatoan_axes

# plotstyle selects a headless (Agg) matplotlib backend on import (it backs the
# file-writing CLIs); re-select inline right after so this hero figure -- and every
# figure below -- renders in the notebook instead of silently going nowhere.
%matplotlib inline

# Shared dark plot style for the policy-building figures below.
FLOOR = "#232c3d"
FLOOR_CMAP = ListedColormap([PAGE, FLOOR])
LABEL_STROKE = [pe.withStroke(linewidth=2.5, foreground=PAGE)]
walk = mesh.grid.to_mask()

def dark_ax(figsize=(12, 6.4)):
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(PAGE)
    ax.set_facecolor(PAGE)
    ax.imshow(walk, cmap=FLOOR_CMAP, interpolation="nearest")
    ax.set_xlim(0, walk.shape[1]); ax.set_ylim(walk.shape[0], 0)
    ax.axis("off")
    return fig, ax

def dark_legend(ax, **kw):
    return ax.legend(frameon=False, labelcolor=SECONDARY, fontsize=8, **kw)

# The establishing shot: the real ship art (desaturated), with everything a policy
# cares about marked on top. Counts and the button rect come from the mesh/world
# model rather than being typed in, so they cannot drift.
fig, ax = plt.subplots(figsize=(12, 6.6))
fig.patch.set_facecolor(PAGE)
croatoan_axes(ax, walk.shape[1], walk.shape[0])
ax.axis("off")
ax.scatter([t.x for t in mesh.tasks], [t.y for t in mesh.tasks], marker="s", s=34,
           facecolors="none", edgecolors=TASK_MARK, linewidths=1.0,
           label=f"task station ({len(mesh.tasks)})", zorder=4)
ax.scatter([v.x for v in mesh.vents], [v.y for v in mesh.vents], marker="D", s=52,
           c=VENT_MARK, edgecolors=EDGE, linewidths=0.6,
           label=f"vent ({len(mesh.vents)})", zorder=5)
bx, by, bw, bh = BUTTON_RECT
ax.scatter([bx + bw / 2], [by + bh / 2], marker="*", s=200, c="#ffd23f",
           edgecolors=EDGE, linewidths=0.6, label="emergency button", zorder=6)
for room, (cx, cy) in mesh.room_centroids().items():
    ax.text(cx, cy - 14, room, color="#cfe6ff", fontsize=8.5, ha="center", va="center",
            weight="bold", path_effects=LABEL_STROKE, zorder=7)
ax.set_title(f"Croatoan: {len(mesh.rooms())} rooms, {len(mesh.tasks)} task stations, "
             f"{len(mesh.vents)} vents, one emergency button",
             color="white", fontsize=12, loc="left")
dark_legend(ax, loc="lower left")
plt.show()


## 2 · Orientation: a policy is just two methods

### How a bot "sees" the game: the `sprite_v1` protocol

Your policy never sees pixels. CrewRift speaks **`sprite_v1`**, a binary **scene-graph**
(display-list) protocol: the server streams *labelled sprite definitions and object
placements with world coordinates*. Identity and position come **for free**, with no computer
vision needed. The `swgy_base` library turns that byte stream into a clean `WorldState` object each
frame, so your job is only to decide what to do.

### The policy contract

`swgy_base` defines the policy interface as a **`typing.Protocol`**. You don't subclass
anything; you implement two methods and you *are* a policy (duck-typed):

```python
class Policy(Protocol):
    def reset(self) -> None: ...          # called once at the start of an episode
    def act(self, world: WorldState) -> Action: ...   # called every frame

class Action:
    buttons: int = 0        # a 7-bit d-pad/A/B button mask
    chat: str | None = None # optional chat message
```

### The runtime loop

`run_player(policy)` connects to the game server (URL from `$COWORLD_PLAYER_WS_URL`) and, for
every inbound frame, runs:

```
sprite_v1 bytes → SceneGraph.apply_packet → read_world → GameTracker.observe → policy.act(world) → button packet
```

`GameTracker` reconstructs the cross-tick state the engine never streams (kill cooldown, task
progress, who your fellow imposters are, …) and stamps it onto the `WorldState` before you see it.

### The `WorldState`: what you read each frame

A few fields you'll use constantly:

- `world.phase`: `LOBBY / ROLE_REVEAL / PLAYING / VOTING / VOTE_RESULT / GAME_OVER`.
- `world.me_collision`: **your true 1×1 collision point** in world pixels. ⚠️ *Always measure
  distances from this, not `me_world`*. The self sprite centre sits `(-4, +2)` px (≈4.5px) off
  the collision point, and mixing frames between yourself and an *other* compounds to a constant
  `(7, 7)` px ≈ 9.9px error, half of the 20px kill range (`docs/COORDINATE_FRAMES.md`).
- `world.others`: the other players you can currently see (each an `Actor` with colour, x, y, alive).
- `world.remaining_task_ks` + `world.task_rect(k)`: your still-to-do tasks and the exact world
  rect to stand in for each.
- `world.me_is_imposter`, `world.kill_ready`, `world.vents`, `world.vote_cursor_on_skip`, …

Let's peek at the mesh's tags to see how much structure comes baked in.

In [ ]:
# (uses `mesh` from the §1 smoke test)
from collections import Counter

tag_counts = Counter(t for n in mesh.nodes for t in n.tags)
print("node tags on the Croatoan mesh:", dict(tag_counts))
print("task stations baked into the mesh:", len(mesh.tasks))
print("visibility baked in? (swgy-navmesh-sight step) ->", mesh.has_visibility)

# A node carries a position and free-form tags; edges are weighted connections.
n0 = mesh.nodes[0]
print("\nexample node:", n0)
print("its neighbours (id, weight, tags):", mesh.neighbors(n0.id)[:3])

### 2b · Contract cheat sheet (for you and your coding assistant)

Skim this on a first read; every row reappears with a worked example in §3. It lives here, up
front, so it's easy to find again and easy to hand to a coding assistant before it writes a
line. Everything in the table is verified against the engine source (`sim.nim`) and
`swgy_base`; when generated code disagrees with a row here, the code is wrong.

| Contract | Common wrong assumption | The truth |
|---|---|---|
| Distances | measure from `me_world`, or hand-apply sprite offsets to others | measure from `world.me_collision`; `others` and `bodies` are already collision-frame |
| Ranges | guessed | kill 20px, report 20px, vent 16px, collision point to collision point |
| Arrival | "close enough" proximity checks | `NavState.heading(...) == (0, 0)` is the only arrival signal; give the follower one destination and let it settle |
| The A button | dedicated kill/report keys exist | A only. Engine priority per press: report a body ≤20px → emergency button (on its pad) → kill (imposter) or task (crew, held while stationary) |
| Vent | any button, any range | B, imposters only, within 16px of a vent centre; teleports to the next vent in the group |
| Presses | held buttons repeat | kill / report / button / vent / vote cursor fire on a *fresh* press; release for a tick between presses |
| Kill readiness | count cooldown ticks yourself | read `world.kill_ready` (the HUD bit is authoritative) |
| Role | `Actor` has a role field | `Actor` is only `color, x, y, alive`. Your own role is `world.me_is_imposter` (latch it; the HUD button hides in meetings); fellow imposters by colour via `world.teammates` |
| Vision | the full roster streams every frame | `world.others` holds only players currently visible to you |
| Voting | a free-form vote API | the cursor starts on the first living candidate, steps one cell per fresh Right press, A confirms; SKIP is the last cell; live tally in `vote_tally` / `vote_skip_count` / `vote_order`; abstaining costs −10 |
| Packaging | "the class is the policy" | a submittable policy is the class plus its helper functions and constants; §3i writes the complete `policy.py` for you |

**What to point your LLM at** when it writes or edits a policy:

- this notebook (the table above plus the worked scenarios in §3g and §3h);
- the `policy.py` that §3i writes: a complete, self-contained, working example;
- inside the installed `swgy_base` wheel: `nav/follow.py` (the follower contract, spelled out in
  its module docstring), `world/model.py` (every `WorldState` field, commented), and
  `runtime/__init__.py` (the play loop and the `Policy` interface).

Then validate whatever it produced with the fuzz harness in §4 before packaging.

## 3 · Build a starter policy, a step at a time

Every round, the engine secretly assigns you **crew** or **imposter**, so a complete policy needs
*both* brains and has to notice which one it's wearing. We'll build exactly that, one piece at a
time, then assemble a single `StarterPolicy` that:

- as **crew**: walks to its tasks and does them, and votes safely (skip, or pile onto a forming
  ejection) to dodge the −10 abstain penalty;
- as **imposter**: hunts an isolated crewmate, kills when in range, vents away, and reports bodies
  to force meetings;
- **detects its role at the start of the round** (`world.me_is_imposter`) and dispatches accordingly.

We start with the crew half, which is the gentler introduction to movement and the world model, then
add the imposter half, then wire both into one policy.

### 3a · The skeleton

The minimum viable policy: load the mesh in `reset()`, return "do nothing" from `act()`.
Note we never subclass `Policy`; implementing the two methods is enough.

In [ ]:
class SkeletonPolicy:
    def reset(self) -> None:
        self.mesh = load_default_mesh()

    def act(self, world: WorldState) -> Action:
        return Action(buttons=0)      # no buttons pressed

p = SkeletonPolicy(); p.reset()
# `Policy` is a runtime_checkable Protocol, so a structural check works:
print("Satisfies the Policy protocol?", isinstance(p, Policy))
print("act() returns:", p.act(WorldState(phase=Phase.LOBBY)))

### 3b · Moving: route over the mesh and follow the plan

Navigation uses `find_path` and `NavState` from `swgy_base.nav`, plus one button helper from
`swgy_base.protocol`:

- **`find_path(mesh, start, goal)`** → an A\* `NavPlan` (a cached list of waypoints). `start`/`goal`
  may be node ids *or* `(x, y)` coordinates (snapped to the nearest node).
- **`NavState`** → an inertia-aware *follower*. You `set_plan(plan)` once, then each tick call
  `update(me)` and `heading(me)` to get a `(dx, dy)` direction.
- **`mask_from_heading(dx, dy)`** (from `swgy_base.protocol`) → turn that direction into a
  wire button mask.

**The follower contract (important):** `NavState` already models the engine's momentum and
*coasts to a stop on the goal*. `heading(...) == (0, 0)` is the **only** "I have arrived" signal.
Give it one destination and act only when it reports `(0, 0)`. The follower handles braking and
settling, so don't fight it with proximity checks or per-tick re-jittering of the destination.

**Somewhere worth going: routing to rooms by name.** You don't invent coordinates or hardcode
strings. `swgy_base.nav` names the map's rooms for you:

- **`CROATOAN_ROOMS`** → the tuple of room names to iterate over; **`CroatoanRoom.REACTOR`** → a
  typo-safe constant you can route to (it's just the string `"Reactor"`). For any mesh (e.g. one you
  rebuild in §7), **`mesh.rooms()`** returns whatever room names it actually carries.
- **`mesh.room_centroid(name)`** → a room's centre (`None` if that room isn't on this mesh);
  **`mesh.tasks`** → every task station as `TaskStation(id, x, y, room, name)`, so "tasks per room"
  is one line and `mesh.task_node(station)` gives the node to route to.

From here the figures switch to the **policy's-eye view**: the two-tone floor is the
walk mask — literally the grid your bot navigates. (The pretty art above is what *we*
see; the mask is what *it* sees.)

A natural first move for a crew bot: from wherever you spawn, head to a **task-rich room** to check
for assigned tasks. We route from a spot on the Bridge to the Reactor's centroid.

In [ ]:
%matplotlib inline
# (uses `mesh` from the §1 smoke test)
import matplotlib.pyplot as plt
from swgy_base.nav import find_path, NavState, NavParams, CroatoanRoom, CROATOAN_ROOMS
from swgy_base.protocol import mask_from_heading

# swgy_base names the map's rooms for you: CROATOAN_ROOMS is the list to iterate, CroatoanRoom.<NAME>
# is a typo-safe constant, and mesh.room_centroid(name) / mesh.tasks turn names into coordinates.
tasks_per_room = Counter(t.room for t in mesh.tasks)
print("rooms you can route to:", list(CROATOAN_ROOMS))
print("task stations per room:", tasks_per_room.most_common(5))

# Crew opener: from your spawn, head to a task-rich room and check for assignments.
SPAWN = (181, 346)                          # a contrived "current position" (a spot on the Bridge)
GOAL_ROOM = CroatoanRoom.REACTOR            # a named constant, not a bare string
goal = mesh.room_centroid(GOAL_ROOM)        # first-class: the room's centre, no averaging by hand
plan = find_path(mesh, SPAWN, goal)         # coords snap to the nearest node
print(f"spawn {SPAWN} -> {GOAL_ROOM} {goal}: {len(plan)} waypoints, cost {plan.cost:.0f}")
print(f"{GOAL_ROOM} holds {tasks_per_room[GOAL_ROOM]} task stations to check for assignments")

# Draw the ship floor, the waypoint graph faintly, and the route on top.
from matplotlib.collections import LineCollection
fig, ax = dark_ax()
segs = [[(mesh.node(e.src).x, mesh.node(e.src).y),
         (mesh.node(e.dst).x, mesh.node(e.dst).y)] for e in mesh.edges]
ax.add_collection(LineCollection(segs, colors="#33415c", linewidths=0.35, zorder=2))
xs = [w[0] for w in plan.waypoints]; ys = [w[1] for w in plan.waypoints]
ax.plot(xs, ys, "-o", color="#ff5533", ms=3.5, lw=2.2, zorder=5, label="planned route")
room_tasks = [t for t in mesh.tasks if t.room == GOAL_ROOM]
ax.scatter([t.x for t in room_tasks], [t.y for t in room_tasks], marker="s", s=80,
           facecolors="none", edgecolors=TASK_MARK, linewidths=1.6, zorder=4,
           label=f"{GOAL_ROOM} task stations")
ax.scatter([SPAWN[0]], [SPAWN[1]], c="#2277ff", s=110, zorder=6, edgecolors="white",
           linewidths=0.7, label="start (spawn)")
ax.scatter([goal[0]], [goal[1]], c="#22dd66", s=120, zorder=6, edgecolors="white",
           linewidths=0.7, label=f"goal: {GOAL_ROOM} centroid")
ax.set_title(f"A* route: spawn -> {GOAL_ROOM}", color="white", fontsize=12, loc="left")
dark_legend(ax, loc="upper left")
plt.show()

# How you'd consume it tick-by-tick:
nav = NavState(params=NavParams(), grid=mesh.grid)
nav.set_plan(plan)
nav.update(SPAWN)
dx, dy = nav.heading(SPAWN)
print(f"first step heading = ({dx}, {dy})  ->  button mask = 0x{mask_from_heading(dx, dy):02x}")

### 3b (bonus) · Route room-to-room with the room graph

`mesh.room_graph()` returns room adjacency as `{room: {neighbours}}`, a sparse map of which
rooms actually connect (on Croatoan, **Storage Deck** is the central hub). It is the nav mesh collapsed
to room granularity (a *quotient* graph): corridor nodes are assigned to their nearest room, so the graph mirrors the ship's real
topology at whatever grid density you built.

That lets a policy **reason in rooms** instead of grid nodes. A BFS over this dict answers
questions a single fine A\* path never exposes: which rooms lie between you and a target, which
room is the ship's chokepoint, whether a region is cut off. It is **not** a cheaper way to
travel — expand a room route back into movement and you get a *longer* path than one direct
A\*, which is what you'd actually follow. The payoff is strategic, over ~13 rooms rather than
thousands of nodes: patrol coverage (the §3f imposter ranks rooms by task count and sweeps
them), cutting off a victim's escape, task allocation. **Decide *where* with the room graph;
walk *there* with one A\*.**

In [ ]:
from collections import Counter, deque
adj = mesh.room_graph()

def room_route(adj, start, goal):
    """Shortest room sequence start -> goal by BFS over the room adjacency graph."""
    start, goal = str(start), str(goal)          # accept CroatoanRoom members or plain names
    prev = {start: None}
    dq = deque([start])
    while dq:
        r = dq.popleft()
        if r == goal:
            break
        for nb in sorted(adj[r]):                 # sorted -> deterministic route
            if nb not in prev:
                prev[nb] = r
                dq.append(nb)
    if goal not in prev:
        return []
    path = [goal]
    while prev[path[-1]] is not None:
        path.append(prev[path[-1]])
    return path[::-1]

# The room graph answers *topology* questions a single fine A* path never exposes:
print("Reactor's neighbours :", ", ".join(sorted(adj[CroatoanRoom.REACTOR])))
hub = max(adj, key=lambda r: len(adj[r]))        # most-connected room == the ship's chokepoint
print(f"busiest junction     : {hub} ({len(adj[hub])} rooms) -- the spot to watch as imposter")
route = room_route(adj, CroatoanRoom.SHUTTLE_BAY, CroatoanRoom.REACTOR)
print("Shuttle Bay->Reactor :", " -> ".join(route), f"({len(route)} rooms to cross)")

# Strategy picks the room; ONE direct A* handles the actual walk (you never stitch centroids --
# that would only make the path longer). Decide WHERE up here, then plan HOW once:
plan = find_path(mesh, mesh.room_centroid(route[0]), mesh.room_centroid(route[-1]))
print(f"target {route[-1]}: one direct A* -> {len(plan)} waypoints (this is the path you follow)")

# Draw the graph the policy reasons over: rooms as nodes (sized by task count), quotient edges.
cents = mesh.room_centroids()
tasks_in = Counter(t.room for t in mesh.tasks)
fig, ax = dark_ax()
drawn = set()
for a, nbrs in adj.items():
    for b in nbrs:
        if (b, a) not in drawn:
            drawn.add((a, b))
            ax.plot([cents[a][0], cents[b][0]], [cents[a][1], cents[b][1]],
                    color="#4f5f82", lw=1.8, alpha=0.9, zorder=3)
for room, (cx, cy) in cents.items():
    ax.scatter([cx], [cy], s=140 + 90 * tasks_in.get(room, 0), c="#1d7fd1",
               edgecolors="#9fd8ff", linewidths=1.2, zorder=5)
    ax.text(cx, cy - 22, room, color="#d7ecff", fontsize=8.5, ha="center",
            weight="bold", path_effects=LABEL_STROKE, zorder=7)
ax.set_title("mesh.room_graph(): how the rooms connect (marker size = task stations)",
             color="white", fontsize=12, loc="left")
plt.show()


### 3c · Doing tasks

Crew progress by standing in a task rect and holding **A** until it completes. `GameTracker`
tells us which tasks are still ours via `world.remaining_task_ks`, and `world.task_rect(k)` gives
the exact world rect. The steps:

1. pick the nearest remaining task,
2. route the **collision point** to its centre,
3. follow the plan; when `heading == (0, 0)` (arrived & at rest), **hold A**.

Because we consistently feed `me_collision` to the follower, the *collision point* lands inside
the rect, which is exactly what the engine tests.

**Where does the assignment come from?** Nothing hands your policy a task list at spawn. Each
tick the server streams — *only to you* — a **task arrow** (a pointer clamped to the screen edge)
for every off-screen assigned task, and a **task bubble** for on-screen ones. The arrow's object
id encodes its task index `k`, and an always-present invisible **marker** gives `k`'s exact rect.
`GameTracker` decodes those ids and *accumulates* them across ticks into `world.remaining_task_ks`
(an arrow disappears the moment you stand on its task, so it has to remember). So a crew policy
learns its whole assignment within a tick or two — by reading arrows, not by wandering to
discover tasks.

> ⚠️ **This crew brain depends entirely on task arrows.** If a game runs with `showTaskArrows`
> off, no arrows stream, `remaining_task_ks` stays empty, `choose_task` returns `None`, and the
> bot idles at spawn. League games have arrows **on**; the local *certification fixture* — which
> is what a bare `coworld run-episode` runs by default — has them **off**, so a crew bot scoring
> zero there is that config, not a broken policy. Run `coworld run-episode --variant default`
> (or the local `run-crewrift.sh` game) to see real crew behaviour.


In [ ]:
from swgy_base.protocol import BTN_A

def choose_task(world, me):
    # Nearest still-assigned task index k, or None.
    best, best_d2 = None, None
    for k in world.remaining_task_ks:
        rect = world.task_rect(k)
        if rect is None:
            continue
        cx, cy = rect[0] + rect[2] // 2, rect[1] + rect[3] // 2
        d2 = (cx - me[0]) ** 2 + (cy - me[1]) ** 2
        if best_d2 is None or d2 < best_d2:
            best, best_d2 = k, d2
    return best

def rect_center(rect):
    return (rect[0] + rect[2] // 2, rect[1] + rect[3] // 2)

# One-line proof it picks sensibly: two tasks, one near, one far -> takes the near one.
toy = WorldState(phase=Phase.PLAYING, map_origin=(0, 0), me_screen=(190, 346),
                 remaining_task_ks=(0, 1),
                 task_markers={0: (210, 350), 1: (964, 427)},
                 task_marker_dims={0: (14, 14), 1: (14, 14)})
print("choose_task picks task", choose_task(toy, toy.me_collision), "(the near one)")

### 3d · Voting: skip safely, or pile onto a forming ejection

The cheapest correct vote is **skip**: it dodges the −10 abstain penalty with no deduction. The
ballot also streams a **live tally**: `world.vote_tally` (colour → votes so far),
`world.vote_skip_count`, and `world.vote_order` (the living candidates in cursor order). A crew bot
can read that and **pile onto a forming ejection** rather than always skipping. An imposter keeps
skipping so it never helps vote out its own partner.

Two mechanics to respect:

- **The cursor is edge-stepped.** It advances one cell per *fresh* Right press (Up/Left = back,
  Down/Right = forward), and **A** confirms the highlighted choice. Holding Right moves it *once*, so
  you must release between presses. `world.vote_cursor_on_skip` is `True` when it rests on SKIP (the
  last cell).
- **Decide once, then commit.** We pick a target when the ballot appears and dead-reckon Right to its
  cell (`vote_order.index(colour)` steps from the first candidate), then confirm. On any uncertainty
  we fall back to SKIP, so we never eject a crewmate by mis-stepping.

`choose_vote` below is the pure decision; the cursor-stepping lives in the policy, which holds the
release-between-presses state.

In [ ]:
def choose_vote(world, min_votes=2):
    """Colour of a forming ejection to pile onto, or None to skip.

    Read the live tally and back the current leader only on a real, unambiguous lead: at least
    `min_votes`, strictly more than SKIP, and strictly ahead of the runner-up. Anything softer
    returns None -> skip, the safe, penalty-free default (never eject on a hunch).
    """
    tally = world.vote_tally
    if not tally:
        return None
    leader, votes = max(tally.items(), key=lambda kv: kv[1])
    runner = max((v for c, v in tally.items() if c != leader), default=0)
    if votes >= min_votes and votes > world.vote_skip_count and votes > runner:
        return leader
    return None

# Three sample ballots: a clear lead -> back it; anything murky -> None (skip).
for tally, skips, expect in [({"red": 3, "lime": 1}, 1, "back red"),
                             ({"red": 2, "lime": 2}, 0, "skip (tied)"),
                             ({"red": 1}, 2, "skip (skip leads)")]:
    w = WorldState(phase=Phase.VOTING, vote_tally=tally, vote_skip_count=skips)
    print(f"tally={tally} skips={skips} -> {choose_vote(w)!r:8} ({expect})")

### 3e · The imposter half: reading your role and the buttons

The crew pieces are done. Now the imposter half. A few things to know about how CrewRift actually
works, and the code stays short:

**1 · How you learn your role.** You don't pick it; the engine assigns it. As an imposter, a **kill
button** rides on your HUD, and `swgy_base` surfaces that as `world.me_is_imposter` (`True` every
PLAYING frame you're an imposter). Crew never see the button, so it stays `False`. We **latch** it on
the first PLAYING frame because the button is hidden during meetings.

**2 · Who's a teammate.** During `ROLE_REVEAL` the engine shows an imposter their fellow imposters;
`GameTracker` records those **colours** in `world.teammates` and carries them through the game. So a
crewmate you may kill is any *alive* `Actor` in `world.others` whose `color` is **not** in
`world.teammates`. (`Actor` only has `color, x, y, alive`, with no `is_imposter` flag to read; the
colour set is how you tell friend from prey.) Others' `(x, y)` are already **collision points** in
world coords, so distances go straight from `world.me_collision`, with no offset math.

**3 · The buttons are overloaded.** There is no dedicated "kill" or "report" key. A single **A**
press is dispatched by the engine in a fixed priority: **report** a body within 20px → press the
**emergency button** if you're standing on its pad → **kill** the nearest crewmate within 20px (or,
for crew, do a task). So *A next to a corpse reports the body; it does not kill*. **B** is the only
other action: **vent** (imposters only), within 16px of a vent, which teleports you to the next vent
in that group. Movement is the d-pad via `mask_from_heading`. One consequence of the priority:
an A press meant as a kill resolves as a *report* whenever a body also sits within 20px, so a
striking imposter should expect that preemption (our policy accounts for it in §3f).

**4 · Presses are edge-triggered.** Kill / report / button / vent all fire on a *fresh* press, the
frame the button goes from up to down. Holding A does nothing after the first frame. So we release
for a tick between presses (`_edge` below). Whether a kill is off cooldown is read from
`world.kill_ready` (the authoritative HUD bit) rather than counting ticks ourselves.

**5 · Reading the map's danger.** The mesh carries baked **line-of-sight**: `mesh.exposure_at(x, y)`
is how open a spot is (`0` hidden … `1` wide open), and `mesh.nodes_by_exposure(...)` filters quiet
vs. open nodes. An imposter uses it to **hunt the most isolated victim**: a crewmate standing where
few others can witness is a safe kill, while a crew bot would rather stay in the open. Our imposter
scores each visible victim by how hidden its spot is (`spot_exposure` below wraps it, with a neutral
fallback when a mesh has no baked sight).

First, the target-picking helpers.

In [ ]:
# Engine action ranges, collision-point to collision-point (from sim.nim).
KILL_RANGE   = 20   # KillRange:   strike a crewmate within this
REPORT_RANGE = 20   # ReportRange: report a body within this
VENT_RANGE   = 16   # VentRange:   enter a vent within this
REPLAN_PX    = 16   # only re-route to a chase target once it drifts past this (anti-thrash)

def nearest_prey(world):
    """Nearest *killable* other (alive, and NOT a fellow imposter) and its distance in px.

    Fellow imposters are known by colour (`world.teammates`), so we skip them. Distances are
    collision-frame: `world.me_collision` vs each other's `(x, y)` (already a collision point),
    so `dist <= KILL_RANGE` matches the engine's real kill test.
    """
    me = world.me_collision
    if me is None:
        return None, None
    best, best_d2 = None, None
    for a in world.living_others():          # alive + currently visible
        if a.color in world.teammates:       # never hunt my own partner
            continue
        d2 = (a.x - me[0]) ** 2 + (a.y - me[1]) ** 2
        if best_d2 is None or d2 < best_d2:
            best, best_d2 = a, d2
    return (best, best_d2 ** 0.5) if best is not None else (None, None)

def nearest_body(world):
    """Nearest visible body (x, y) and its distance in px, or (None, None)."""
    me = world.me_collision
    if me is None or not world.bodies:
        return None, None
    b = min(world.bodies, key=lambda p: (p[0] - me[0]) ** 2 + (p[1] - me[1]) ** 2)
    return b, ((b[0] - me[0]) ** 2 + (b[1] - me[1]) ** 2) ** 0.5

def nearest_vent(world):
    """Nearest vent centre (x, y) and its distance in px, or (None, None)."""
    me = world.me_collision
    if me is None or not world.vents:
        return None, None
    v = min(world.vents, key=lambda p: (p[0] - me[0]) ** 2 + (p[1] - me[1]) ** 2)
    return v, ((v[0] - me[0]) ** 2 + (v[1] - me[1]) ** 2) ** 0.5

def spot_exposure(mesh, pos):
    """How exposed a point is: 0 (hidden) .. 1 (wide open), from the mesh's baked line-of-sight.
    Falls back to 0.5 when the mesh carries no visibility data (e.g. one rebuilt from a walk grid)."""
    if pos is None or not mesh.has_visibility:
        return 0.5
    v = mesh.exposure_at(int(pos[0]), int(pos[1]))
    return 0.5 if v is None else v   # a single node can lack baked sight on a custom mesh

# Proof the teammate filter works: partner adjacent, crew farther -> hunts the crew.
from swgy_base.world import Actor
toy = WorldState(phase=Phase.PLAYING, map_origin=(0, 0), me_screen=(200, 350),
                 teammates={"red"},
                 others=[Actor("red", 206, 352, True), Actor("lime", 260, 352, True)])
prey, dist = nearest_prey(toy)
print(f"nearest_prey skips the partner (red) -> hunts {prey.color} at {dist:.0f}px | "
      f"ranges kill/report/vent = {KILL_RANGE}/{REPORT_RANGE}/{VENT_RANGE}px")

### 3f · Assemble the `StarterPolicy`: one policy, both roles

Now wire everything into a single policy. `act()` dispatches on **phase** and **role**:

- **VOTING** → `_vote`: crew reads the live tally and **piles onto a forming ejection** (else skips);
  the imposter always skips, so it never helps vote out its own partner.
- **PLAYING** → latch the role from `world.me_is_imposter`, then run `_play_crew` (the
  task-runner from §3c) or `_play_imposter`.
- everything else (LOBBY / ROLE_REVEAL / VOTE_RESULT / GAME_OVER) → sit tight.

The imposter brain is a short priority list, checked top to bottom each tick:

1. **just killed?** → flee into the nearest **vent** (`B`) to leave the scene. This also keeps us
   from standing on our fresh corpse and *reporting* it by accident on the next `A`;
2. **a crewmate in kill range and the kill bar is full?** → strike (`A`), then set the flee flag;
3. **a body within reach?** → report it (`A`) to force a meeting we can steer;
4. **can see a victim?** → close on the most **isolated** one (lowest `spot_exposure`, so few
   others can witness the kill; distance breaks ties). We *commit* to one victim colour rather
   than re-picking every tick, so we don't ping-pong between crewmates;
5. **nobody in sight?** → **patrol** the task-dense rooms where crew gather. A stationary imposter
   finds no one *and* bleeds the engine's −1-per-20s idle penalty, so when the coast is clear we
   keep moving toward where the kills are.

It still only *hunts* players it can currently see (the engine streams only visible others) — but
it no longer freezes when it sees nobody. The "Next steps" note points at the richer version
(memory-backed pursuit of a victim who slips out of view).


In [ ]:
from swgy_base.protocol import BTN_A, BTN_B, BTN_RIGHT, mask_from_heading
from swgy_base.nav import find_path, NavState, NavParams, load_default_mesh

class StarterPolicy:
    # Role-aware starter: run tasks as crew; hunt/kill/vent/report as imposter;
    # vote by reading the live tally (crew) or skipping safely (imposter).

    def reset(self) -> None:
        self.mesh = load_default_mesh()
        self.nav = NavState(params=NavParams(), grid=self.mesh.grid)
        # Imposter patrol: with no prey visible, sweep the task-dense rooms (where crew
        # gather) instead of idling and bleeding the -1/20s stuck penalty. Rank rooms by
        # how many task stations they hold; loop through their centroids.
        room_tasks = {}
        for t in self.mesh.tasks:
            if t.room:
                room_tasks[t.room] = room_tasks.get(t.room, 0) + 1
        cents = self.mesh.room_centroids()
        self._patrol_pts = [cents[r] for r in sorted(room_tasks, key=room_tasks.get, reverse=True)
                            if r in cents]
        self._patrol_i = 0
        self._target_k = None       # crew: task I'm currently walking to
        self._am_imposter = False   # latched once I see my kill button (it hides in meetings)
        self._victim = None         # imposter: committed pursuit target (a colour)
        self._fleeing = False       # imposter: just killed -> head for a vent
        self._nav_goal = None       # imposter: point the current plan was routed to
        self._last_press = 0        # edge-trigger state for A/B/RIGHT presses
        self._vote_decided = False  # voting: decide the target once per meeting
        self._vote_target = None    # colour to vote for, or None to skip
        self._vote_steps = 0        # RIGHT presses left to reach the target cell

    def act(self, world: WorldState) -> Action:
        if world.phase is Phase.PLAYING and world.me_is_imposter:
            self._am_imposter = True                 # latch role (the button hides in meetings)
        if world.phase is not Phase.VOTING:
            self._vote_decided = False               # fresh decision each new meeting
        if world.phase is Phase.VOTING:
            return self._vote(world)
        if world.phase is Phase.PLAYING:
            return self._play_imposter(world) if self._am_imposter else self._play_crew(world)
        return Action(buttons=0)                     # LOBBY / ROLE_REVEAL / VOTE_RESULT / GAME_OVER

    # --- crew: walk to the nearest task and hold A (the §3c logic) -----------
    def _play_crew(self, world: WorldState) -> Action:
        me = world.me_collision
        if me is None:
            return Action(buttons=0)
        k = choose_task(world, me)
        if k is None:
            return Action(buttons=0)
        goal = rect_center(world.task_rect(k))
        if k != self._target_k or self.nav.plan is None or self.nav.needs_replan:
            self.nav.set_plan(find_path(self.mesh, me, goal, self.nav.params))
            self._target_k = k
        self.nav.update(me)
        dx, dy = self.nav.heading(me, collision=me)
        if (dx, dy) == (0, 0):
            return Action(buttons=BTN_A)             # arrived & at rest -> hold A to do the task
        return Action(buttons=mask_from_heading(dx, dy))

    # --- imposter: flee -> kill -> report -> hunt the most isolated victim ----
    def _play_imposter(self, world: WorldState) -> Action:
        me = world.me_collision
        if me is None:
            return Action(buttons=0)

        if self._fleeing:                            # just killed -> slip into a vent to escape
            v, vd = nearest_vent(world)
            if v is None:
                self._fleeing = False
            elif vd <= VENT_RANGE:
                self._fleeing = False
                return Action(buttons=self._edge(BTN_B))
            else:
                return self._go_to(me, v)

        body, bd = nearest_body(world)               # a nearby body changes what an A press does

        near, ndist = nearest_prey(world)            # kill any crewmate in range while ready
        if near is not None and ndist <= KILL_RANGE and world.kill_ready:
            # The engine resolves A report-first: a body within 20px preempts the kill and
            # starts a meeting instead. Only plan the post-kill flee when the kill can land.
            self._fleeing = body is None or bd > REPORT_RANGE
            return Action(buttons=self._edge(BTN_A))

        if body is not None and bd <= REPORT_RANGE:  # stumbled on a body -> report it (A)
            return Action(buttons=self._edge(BTN_A))

        victim = self._pick_victim(world)            # a visible victim -> close on it
        if victim is not None:
            return self._go_to(me, (victim.x, victim.y))
        return self._patrol_step(me)                 # no prey in sight -> roam, don't idle

    def _pick_victim(self, world: WorldState):
        """Commit to one victim: prefer the most ISOLATED visible crewmate (low exposure = a
        safer kill), with distance as the tie-break. Keep the commitment while it stays visible."""
        me = world.me_collision
        if self._victim is not None:                 # honour the existing commitment
            for a in world.living_others():
                if a.color == self._victim and a.color not in world.teammates:
                    return a
            self._victim = None                      # died / left view / turned out a teammate
        best, best_score = None, None
        for a in world.living_others():
            if a.color in world.teammates:
                continue
            d = ((a.x - me[0]) ** 2 + (a.y - me[1]) ** 2) ** 0.5
            score = spot_exposure(self.mesh, (a.x, a.y)) + d / 1000.0   # isolation first, then near
            if best_score is None or score < best_score:
                best, best_score = a, score
        if best is not None:
            self._victim = best.color
        return best

    def _go_to(self, me, target) -> Action:
        target = (int(target[0]), int(target[1]))
        drifted = (self._nav_goal is None
                   or (target[0] - self._nav_goal[0]) ** 2
                   + (target[1] - self._nav_goal[1]) ** 2 > REPLAN_PX ** 2)
        if self.nav.plan is None or self.nav.needs_replan or drifted:
            self.nav.set_plan(find_path(self.mesh, me, target, self.nav.params))
            self._nav_goal = target
        self.nav.update(me)
        dx, dy = self.nav.heading(me, collision=me)
        return Action(buttons=mask_from_heading(dx, dy))

    def _patrol_step(self, me) -> Action:
        """No prey visible: sweep the task-dense rooms so we keep moving (dodging the
        -1/20s idle penalty) and cross paths with crew. Hop to the next room once close."""
        if not self._patrol_pts:
            return Action(buttons=0)
        goal = self._patrol_pts[self._patrol_i]
        if (goal[0] - me[0]) ** 2 + (goal[1] - me[1]) ** 2 <= 48 ** 2:   # close -> next room
            self._patrol_i = (self._patrol_i + 1) % len(self._patrol_pts)
            goal = self._patrol_pts[self._patrol_i]
        return self._go_to(me, goal)

    def _edge(self, mask: int) -> int:
        """Fire `mask` only on a fresh press: release for one tick between presses so the engine
        sees a rising edge (kill/report/vent AND the vote cursor all act on the edge, not the hold)."""
        out = 0 if mask == self._last_press else mask
        self._last_press = out
        return out

    # --- voting: crew piles onto a forming ejection; imposter skips ----------
    def _vote(self, world: WorldState) -> Action:
        if self._am_imposter:
            return self._step_to_skip(world)         # never risk a teammate
        if not self._vote_decided:                   # crew: decide once, then commit
            t = choose_vote(world)
            self._vote_target = t if (t is not None and t in world.vote_order) else None
            self._vote_steps = world.vote_order.index(self._vote_target) if self._vote_target else 0
            self._vote_decided = True
        if self._vote_target is None:
            return self._step_to_skip(world)         # no clear ejection -> skip
        if self._vote_steps > 0:                     # dead-reckon RIGHT to the target's cell
            out = self._edge(BTN_RIGHT)
            if out:
                self._vote_steps -= 1
            return Action(buttons=out)
        return Action(buttons=self._edge(BTN_A))     # on the target cell -> confirm

    def _step_to_skip(self, world: WorldState) -> Action:
        # Walk the cursor to SKIP (the last cell) and confirm; edge-triggered so each press lands.
        return Action(buttons=self._edge(BTN_A if world.vote_cursor_on_skip else BTN_RIGHT))

policy = StarterPolicy(); policy.reset()
print("StarterPolicy ready. Satisfies Policy protocol?", isinstance(policy, Policy))

### 3g · Exercise it offline: both roles

No game server needed: we hand-build `WorldState` snapshots and call `act()` directly. We put the
map origin at `(0, 0)` so world coords == screen coords, drop ourselves on the map with `me_screen`,
and set the role and threat fields for each scenario. (`me_collision` is derived from `me_screen` and
the map origin, so you don't set it directly.) We drive both brains: the crew task run; the imposter scenarios (pursuit, a kill in range, a
spared teammate, a report, and isolation-based target choice); and four ballots exercising the
tally voting.

In [ ]:
from swgy_base.world import Actor

def decode_mask(m):
    from swgy_base.protocol import BTN_UP, BTN_DOWN, BTN_LEFT, BTN_RIGHT, BTN_A, BTN_B
    names = [("UP", BTN_UP), ("DOWN", BTN_DOWN), ("LEFT", BTN_LEFT),
             ("RIGHT", BTN_RIGHT), ("A", BTN_A), ("B", BTN_B)]
    return "+".join(n for n, b in names if m & b) or "(none)"

# Imposter scenarios sit at me_screen=(200, 350) with origin (0, 0) -> me_collision == (196, 352);
# the crew scenario starts nearer the Bridge, at me_screen=(190, 346).

# CREW: me_is_imposter is False -> crew brain. One task far east -> walk toward it.
crew = WorldState(phase=Phase.PLAYING, map_origin=(0, 0), me_screen=(190, 346),
                  remaining_task_ks=(0,), task_markers={0: (964, 427)}, task_marker_dims={0: (14, 14)})
p = StarterPolicy(); p.reset()
a = p.act(crew)
print(f"CREW  · task far east      -> {decode_mask(a.buttons):10}  (walking to the task)")

# IMPOSTER · pursue: a crewmate ~64px away, out of kill range -> route toward it.
imp = StarterPolicy(); imp.reset()
pursue = WorldState(phase=Phase.PLAYING, map_origin=(0, 0), me_screen=(200, 350),
                    me_is_imposter=True, kill_ready=True, others=[Actor("red", 260, 352, True)])
a = imp.act(pursue)
print(f"IMP   · crew 64px away     -> {decode_mask(a.buttons):10}  (pursuing)")

# IMPOSTER · kill: crewmate ~10px away and kill bar ready -> strike (A).
imp.reset()
close = WorldState(phase=Phase.PLAYING, map_origin=(0, 0), me_screen=(200, 350),
                   me_is_imposter=True, kill_ready=True, others=[Actor("red", 206, 352, True)])
a = imp.act(close)
print(f"IMP   · crew 10px, ready   -> {decode_mask(a.buttons):10}  (kill)")

# IMPOSTER · spare a teammate: the only nearby player is my partner.
imp.reset()
mate = WorldState(phase=Phase.PLAYING, map_origin=(0, 0), me_screen=(200, 350),
                  me_is_imposter=True, kill_ready=True, teammates={"red"},
                  others=[Actor("red", 206, 352, True)])
a = imp.act(mate)
print(f"IMP   · only a teammate    -> {decode_mask(a.buttons):10}  (never target a partner)")

# IMPOSTER · report: a body right next to me -> report it (A).
imp.reset()
corpse = WorldState(phase=Phase.PLAYING, map_origin=(0, 0), me_screen=(200, 350),
                    me_is_imposter=True, bodies=[(196, 352)])
a = imp.act(corpse)
print(f"IMP   · body in reach      -> {decode_mask(a.buttons):10}  (report -> meeting)")

# IMPOSTER · isolation: two crew visible -- one in the open (Bridge), one in a quiet corner.
imp.reset()
iso = WorldState(phase=Phase.PLAYING, map_origin=(0, 0), me_screen=(300, 300), me_is_imposter=True,
                 others=[Actor("red", 185, 346, True),     # Bridge: wide open (high exposure)
                         Actor("lime", 448, 154, True)])   # quiet corner (low exposure)
victim = imp._pick_victim(iso)
print(f"IMP   · isolate target     -> {victim.color:10}  (hunts the least-witnessed crewmate)")

# IMPOSTER · nobody in sight -> patrol a task-dense room instead of idling (no idle penalty).
imp.reset()
empty = WorldState(phase=Phase.PLAYING, map_origin=(0, 0), me_screen=(300, 300),
                   me_is_imposter=True)          # others=[] -> no prey visible
a = imp.act(empty)
print(f"IMP   · no one visible     -> {decode_mask(a.buttons):10}  (patrol toward crew-heavy rooms)")

# VOTE · crew piles onto a forming ejection (red leads 3-1) -> confirm on red (the first cell).
cb = StarterPolicy(); cb.reset()
eject = WorldState(phase=Phase.VOTING, vote_cursor_on_skip=False,
                   vote_tally={"red": 3, "lime": 1}, vote_skip_count=1, vote_order=("red", "lime"))
a = cb.act(eject)
print(f"VOTE  · crew, red leads    -> {decode_mask(a.buttons):10}  (confirm the ejection)")

# VOTE · target is the 2nd cell -> step RIGHT, then confirm (two ticks).
cb.reset()
eject2 = WorldState(phase=Phase.VOTING, vote_cursor_on_skip=False,
                    vote_tally={"lime": 3, "red": 1}, vote_skip_count=1, vote_order=("red", "lime"))
a1 = cb.act(eject2); a2 = cb.act(eject2)
print(f"VOTE  · lime leads (2nd)   -> {decode_mask(a1.buttons)} then {decode_mask(a2.buttons)}  (step, then confirm)")

# VOTE · no clear lead -> crew skips (RIGHT toward SKIP).
cb.reset()
noej = WorldState(phase=Phase.VOTING, vote_cursor_on_skip=False,
                  vote_tally={"red": 1}, vote_skip_count=2, vote_order=("red", "lime"))
a = cb.act(noej)
print(f"VOTE  · crew, no lead      -> {decode_mask(a.buttons):10}  (head for SKIP)")

# VOTE · imposter always skips, even with a lead forming (never out a partner).
imp.reset(); imp._am_imposter = True
a = imp.act(WorldState(phase=Phase.VOTING, vote_cursor_on_skip=True,
                       vote_tally={"red": 3}, vote_skip_count=0, vote_order=("red",)))
print(f"VOTE  · imposter on SKIP   -> {decode_mask(a.buttons):10}  (skip; don't out a teammate)")

print("\nOffline checks pass: one policy, both roles, visibility-aware hunting + tally-aware voting.")

In [ ]:
# The payoff: watch the crew brain MOVE. We drive its chooser + follower through
# swgy_tools' byte-faithful port of the engine physics (real acceleration, friction,
# wall-sliding -- §8 explains this simulator; here we just borrow it), recording the
# collision point every tick. No game server.
from swgy_tools.navbench.enginesim import Engine, Player
from swgy_tools.plotstyle import ALIVE, DEATH

assigned = [next(t for t in mesh.tasks if t.room == room)
            for room in ("Bridge", "Science Bay", "Storage Deck")]

eng, pl = Engine(mesh.grid), Player(x=181, y=346)
nav = NavState(params=NavParams(), grid=mesh.grid)
remaining = {t.id: t for t in assigned}
trail, done, goal_k = [(pl.x, pl.y)], [], None
for tick in range(6000):
    if not remaining:
        break
    me = (pl.x, pl.y)
    k = min(remaining, key=lambda k: (remaining[k].x - me[0]) ** 2 + (remaining[k].y - me[1]) ** 2)
    if k != goal_k:
        nav.set_plan(find_path(mesh, me, (remaining[k].x, remaining[k].y)))
        goal_k = k
    nav.update(me)
    dx, dy = nav.heading(me, collision=me)
    if (dx, dy) == (0, 0):                      # arrived & settled: the hold would start here
        done.append((tick, remaining.pop(k)))
        goal_k = None
        continue
    eng.step(pl, dx, dy)
    trail.append((pl.x, pl.y))

travel = done[-1][0]
total_s = (travel + 72 * len(done)) / 24        # + the engine's 72-tick hold per task
fig, ax = dark_ax()
xs, ys = zip(*trail)
ax.plot(xs, ys, color=ALIVE, lw=1.8, alpha=0.95, zorder=4, label="the policy's actual path")
ax.scatter([181], [346], s=130, facecolors="none", edgecolors="white", linewidths=1.6,
           zorder=6, label="spawn")
for i, (tick, t) in enumerate(done, 1):
    ax.scatter([t.x], [t.y], marker="*", s=240, c="#ffd23f", edgecolors=EDGE,
               linewidths=0.6, zorder=6)
    ax.annotate(f"{i}: {t.room} @ {tick / 24:.0f}s", (t.x, t.y),
                xytext=(8, -12), textcoords="offset points",
                color="white", fontsize=9, path_effects=LABEL_STROKE, zorder=7)
ax.set_title(f"your policy opens the game: {len(done)} tasks in ~{total_s:.0f}s "
             f"({travel} travel ticks + the 72-tick holds) -- simulated engine physics",
             color="white", fontsize=12, loc="left")
dark_legend(ax, loc="lower left")
plt.show()


In [ ]:
# What isolation targeting sees: the mesh's baked line-of-sight as HEAT over the real
# ship. Bright = watched ground; dark = the quiet corners an imposter hunts in. Then
# the two candidates from the scenario above, and the pick.
import numpy as np
from swgy_tools.plotstyle import PAGE, croatoan_axes

CELL_E = 8
gh, gw = (walk.shape[0] + CELL_E - 1) // CELL_E, (walk.shape[1] + CELL_E - 1) // CELL_E
coords = np.array([(n.x, n.y) for n in mesh.nodes], dtype=float)
expo = np.array([0.5 if n.exposure is None else n.exposure for n in mesh.nodes])
heat = np.zeros((gh, gw))
for gy in range(gh):                     # nearest-node exposure per 8px cell (row-wise argmin)
    cy = gy * CELL_E + CELL_E // 2
    cxs = np.arange(gw) * CELL_E + CELL_E // 2
    d2 = (coords[:, 0][None, :] - cxs[:, None]) ** 2 + (coords[:, 1][None, :] - cy) ** 2
    heat[gy] = expo[np.argmin(d2, axis=1)]
walk8 = walk[::CELL_E, ::CELL_E][:gh, :gw]           # hide cells with no floor under them
rgba = plt.get_cmap("inferno")(heat)
rgba[..., 3] = np.where(walk8, 0.55, 0.0)

fig, ax = plt.subplots(figsize=(12, 6.4))
fig.patch.set_facecolor(PAGE)
croatoan_axes(ax, walk.shape[1], walk.shape[0])
ax.axis("off")
ax.imshow(rgba, extent=[0, walk.shape[1], walk.shape[0], 0], interpolation="bilinear", zorder=3)
me_xy, red_xy, lime_xy = (300, 300), (185, 346), (448, 154)
ax.scatter(*me_xy, marker="X", s=170, c="#ff3355", edgecolors="white", linewidths=1.0,
           zorder=6, label="imposter (you)")
ax.scatter(*red_xy, s=120, c="#ff5533", edgecolors="white", linewidths=1.0, zorder=6,
           label="red: open ground")
ax.scatter(*lime_xy, s=120, c="#7CFC00", edgecolors="white", linewidths=1.0, zorder=6,
           label="lime: quiet corner")
ax.add_patch(plt.Circle(lime_xy, 34, fill=False, color="white", lw=1.6, ls="--", zorder=7))
ax.annotate("", xy=lime_xy, xytext=me_xy,
            arrowprops=dict(arrowstyle="-|>", color="white", lw=1.6), zorder=6)
sm = plt.cm.ScalarMappable(cmap="inferno", norm=plt.Normalize(0, 1))
cb = fig.colorbar(sm, ax=ax, fraction=0.026, pad=0.01)
cb.set_label("baked exposure (0 hidden .. 1 wide open)", color="white", fontsize=9)
cb.ax.tick_params(colors="white", labelsize=8)
cb.outline.set_visible(False)
ax.set_title("the imposter's eye: hunt where nobody is watching", color="white",
             fontsize=12, loc="left")
dark_legend(ax, loc="lower left")
plt.show()


### 3h · Your first upgrade: react to what you saw

`GameTracker` reconstructs events the engine never streams and stamps them onto each snapshot:
`world.witnessed_kill` (someone dropped in your view this frame), `world.suspicious` (colours you
personally saw vanish into a vent), and `world.button_calls_remaining` (your emergency-meeting
calls left, one per game; the pad is the `BUTTON_RECT` rect on the Bridge).

`DetectivePolicy` subclasses `StarterPolicy` and reacts as crew. Seeing a kill latches an alarm;
while alarmed it reports the corpse if it can reach one, and otherwise walks to the emergency
button and spends its call. At the ballot it votes a colour it saw vent before falling back to
the tally pile-on. Subclassing is the intended extension pattern: override the phase handler you
want to change and inherit the rest.

In [ ]:
from swgy_base.world import BUTTON_RECT

class DetectivePolicy(StarterPolicy):
    """StarterPolicy that acts on witnessed events (crew side only).

    Reads the GameTracker-reconstructed fields: witnessed_kill latches an alarm,
    suspicious steers the vote, button_calls_remaining gates the emergency call.
    """

    def reset(self) -> None:
        super().reset()
        self._alarm = False          # latched: I saw a kill and want a meeting

    def act(self, world: WorldState) -> Action:
        if world.phase is Phase.VOTING:
            self._alarm = False      # the meeting I wanted is happening
        return super().act(world)

    def _play_crew(self, world: WorldState) -> Action:
        me = world.me_collision
        if me is not None and world.witnessed_kill:
            self._alarm = True                       # remember it; the corpse may be far away
        if me is not None and self._alarm:
            body, bd = nearest_body(world)
            if body is not None:                     # a corpse I can reach: report it
                if bd <= REPORT_RANGE:
                    return Action(buttons=self._edge(BTN_A))
                return self._go_to(me, body)
            if world.button_calls_remaining > 0:     # no corpse in view: raise the alarm myself
                bx, by, bw, bh = BUTTON_RECT
                if bx <= me[0] < bx + bw and by <= me[1] < by + bh:
                    return Action(buttons=self._edge(BTN_A))   # on the pad: call the meeting
                return self._go_to(me, (bx + bw // 2, by + bh // 2))
            self._alarm = False                      # nothing I can do about it: back to tasks
        return super()._play_crew(world)

    def _vote(self, world: WorldState) -> Action:
        # First-hand evidence outranks the tally: vote a colour I saw vent, if it's on the ballot.
        if not self._am_imposter and not self._vote_decided:
            sus = next((c for c in world.vote_order if c in world.suspicious), None)
            if sus is not None:
                self._vote_target = sus
                self._vote_steps = world.vote_order.index(sus)
                self._vote_decided = True
        return super()._vote(world)

det = DetectivePolicy(); det.reset()
print("DetectivePolicy ready. Satisfies Policy protocol?", isinstance(det, Policy))

In [ ]:
# Witnessed a kill with the body in reach -> report it (A).
det.reset()
saw = WorldState(phase=Phase.PLAYING, map_origin=(0, 0), me_screen=(200, 350),
                 witnessed_kill=True, bodies=[(196, 352)])
a = det.act(saw)
print(f"DET · saw a kill, body near   -> {decode_mask(a.buttons):10}  (report)")

# Alarmed, no body in view, call available, standing on the button pad -> call the meeting (A).
det.reset(); det._alarm = True
on_pad = WorldState(phase=Phase.PLAYING, map_origin=(0, 0), me_screen=(185, 344),
                    button_calls_remaining=1)
a = det.act(on_pad)
print(f"DET · alarmed, on the pad     -> {decode_mask(a.buttons):10}  (emergency meeting)")

# Alarmed far from the pad -> walks toward it.
det.reset(); det._alarm = True
far = WorldState(phase=Phase.PLAYING, map_origin=(0, 0), me_screen=(600, 350),
                 button_calls_remaining=1)
a = det.act(far)
print(f"DET · alarmed, far from pad   -> {decode_mask(a.buttons):10}  (heading to the button)")

# On the ballot, first-hand evidence outranks the tally: lime vented in view, red merely leads.
det.reset()
ballot = WorldState(phase=Phase.VOTING, vote_cursor_on_skip=False,
                    vote_tally={"red": 3}, vote_skip_count=0,
                    vote_order=("red", "lime"), suspicious={"lime"})
a1 = det.act(ballot); a2 = det.act(ballot)
print(f"DET · venter on the ballot    -> {decode_mask(a1.buttons)} then {decode_mask(a2.buttons)}  (vote the venter over the pile-on)")

### 3i · Your policy as a file

`StarterPolicy` is a class plus the helper functions and constants defined across §3. Packaging
(§6) needs all of that in one importable file. Rather than asking you to copy cells by hand,
which drifts the moment any cell is edited, the cell below reads this notebook from disk, lifts
the named definitions out of its code cells with `ast`, and writes them to
`my_starter_policy/policy.py`. It then re-imports the file and instantiates both classes as a
smoke test. The result is the canonical artifact the rest of the notebook validates (§4) and
packages (§6), and the best single file to show a coding assistant.

> ⚠️ **Save the notebook before running this cell** (it assembles from the file on
> disk), and re-run it after editing any policy cell. If you renamed the notebook,
> set `NB_PATH` in the cell.

It writes a complete, *buildable* package — `pyproject.toml`, the package, and a
`__main__` runner — so §5 can run it and §6 can build it with no hand-created files.

In [ ]:
import ast
import importlib.util
import json
from pathlib import Path

NB_PATH = Path("LetsPlayCrewrift.ipynb")   # set this if you renamed the notebook

WANTED = ["choose_task", "rect_center", "choose_vote", "nearest_prey", "nearest_body",
          "nearest_vent", "spot_exposure", "StarterPolicy", "DetectivePolicy"]

HEADER = '''"""CrewRift starter policy, assembled by LetsPlayCrewrift.ipynb (§3i)."""
from swgy_base.nav import find_path, NavState, NavParams, load_default_mesh
from swgy_base.protocol import BTN_A, BTN_B, BTN_RIGHT, mask_from_heading
from swgy_base.runtime import Action
from swgy_base.world import BUTTON_RECT, Phase, WorldState

KILL_RANGE   = 20   # engine KillRange, collision frame
REPORT_RANGE = 20   # engine ReportRange
VENT_RANGE   = 16   # engine VentRange
REPLAN_PX    = 16   # chase-target drift before re-routing
'''

MAIN_PY = '''"""Entry point: `python -m my_starter_policy` (the platform sets the URL)."""
from swgy_base.runtime import run_player

from my_starter_policy.policy import StarterPolicy


def main() -> None:
    run_player(StarterPolicy())     # resolves $COWORLD_PLAYER_WS_URL


if __name__ == "__main__":
    main()
'''

PYPROJECT = '''[project]
name = "my-starter-policy"
version = "0.1.0"
requires-python = ">=3.11"
dependencies = ["swgy-base>=0.3.0"]

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[tool.hatch.build.targets.wheel]
packages = ["my_starter_policy"]
'''

# Lift each wanted def/class out of this notebook's code cells (save the notebook first).
if not NB_PATH.exists():
    raise FileNotFoundError(
        f"{NB_PATH} not found in {Path.cwd()} -- if you renamed or moved the notebook, "
        "point NB_PATH at it and re-run this cell."
    )
notebook = json.loads(NB_PATH.read_text())
found = {}
for c in notebook["cells"]:
    if c["cell_type"] != "code":
        continue
    cell_src = "".join(c["source"])
    try:
        tree = ast.parse(cell_src)
    except SyntaxError:
        continue  # cells with % magics don't parse; none of them define policy code
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.ClassDef)) and node.name in WANTED:
            found[node.name] = ast.get_source_segment(cell_src, node)
missing = [name for name in WANTED if name not in found]
assert not missing, f"definitions not found in the notebook (did you save?): {missing}"

# A standard, buildable layout: project root with pyproject, the package inside it.
root = Path("my_starter_policy")
pkg = root / "my_starter_policy"
pkg.mkdir(parents=True, exist_ok=True)
(root / "pyproject.toml").write_text(PYPROJECT)
(pkg / "__init__.py").write_text("")
(pkg / "__main__.py").write_text(MAIN_PY)
(pkg / "policy.py").write_text(HEADER + "\n\n" + "\n\n".join(found[n] for n in WANTED) + "\n")

# Smoke-test the file exactly as a packaged image would use it: import it fresh from disk.
spec = importlib.util.spec_from_file_location("my_starter_policy.policy", pkg / "policy.py")
filemod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(filemod)
ok = isinstance(filemod.StarterPolicy(), Policy) and isinstance(filemod.DetectivePolicy(), Policy)
print(f"wrote {root}/: pyproject.toml + {pkg.name}/{{__init__,__main__,policy}}.py "
      f"({(pkg / 'policy.py').stat().st_size} bytes); fresh import satisfies Policy: {ok}")


**Hand your coding agent** `my_starter_policy/my_starter_policy/policy.py` plus the §2b
contract table, and a prompt like:

> *"Extend StarterPolicy with memory-backed pursuit: remember each visible player's
> last-seen position; when my committed victim leaves view, chase that point instead of
> giving up. Keep the `reset`/`act` signature and every contract in the table (collision
> frame, edge-triggered presses, `heading == (0,0)` as the only arrival signal). Then
> fuzz it exactly as §4 does before you show me the diff."*

The fuzz harness in §4 is the referee for whatever comes back.


> **What you've got, and where to take it.** You now hold a complete, submittable baseline:
> `StarterPolicy` detects its role, runs tasks as crew, hunts the most isolated victim as
> imposter (kill, vent away, report), and votes off the live tally; `DetectivePolicy` layers
> witnessed-event reactions on top; and `my_starter_policy/policy.py` carries the whole thing as
> one file. Ideas the reference players build on this same `swgy_base` foundation:
>
> - **committed, memory-backed pursuit**: chase a victim toward its *last-seen* spot after it
>   leaves view (the engine only streams players you can see), so a corner doesn't end the hunt;
> - **vent-group hopping**: press `B` repeatedly to hop around a vent group and reposition
>   between kills;
> - **a neural brain**: a small numpy MLP maps an egocentric observation to a macro
>   (kill / vent / report / hunt / go-task), with these scripted moves as the fallback;
> - **chat**: `Action.chat` sends a short message each tick if you set it
>   (`swgy_base.protocol.pack_chat_packet` shows the wire rules); misdirection is a craft of
>   its own.
>
> Same two methods (`reset` / `act`), with more happening inside `act`. For worked examples,
> browse the reference players in
> [coworld-crewrift](https://github.com/Metta-AI/coworld-crewrift) (`players/`).

## 4 · Validate: fuzz your policy offline

A submitted policy runs unattended, and a single uncaught exception on an awkward frame (a
ballot with no cursor yet, a frame with no map object) forfeits the round. Before packaging
anything, and especially before packaging code an LLM wrote, drive it through
`swgy_base.testing.fuzz_policy`: hundreds of seeded synthetic snapshots spanning every phase and
the `None`-shaped corners of the world model, checking that `act()` always returns an `Action`
and never raises. The run is deterministic per seed, so any failure reproduces exactly, and the
report carries the offending world summary with its traceback.

We fuzz the notebook classes and the re-imported `policy.py` copy, so the packaging path itself
is covered.

In [ ]:
from swgy_base.testing import fuzz_policy

for cls in (StarterPolicy, DetectivePolicy, filemod.StarterPolicy):
    origin = "policy.py" if cls.__module__ == "my_starter_policy.policy" else "notebook"
    print(f"{cls.__name__:16} [{origin:9}]  {fuzz_policy(cls(), n=500, seed=42)}")

## 5 · Run it live against a real game (*optional, needs Docker*)

To watch the policy actually play, point `run_player` at a running CrewRift server. The engine
assigns your role at the start of each round; `StarterPolicy` handles either one, so you submit the
same class regardless. In production the platform sets `$COWORLD_PLAYER_WS_URL` for you and your
process just plays one episode and exits.

We don't launch a server from the notebook (it would block on a socket). The recipe below was
run end-to-end on a stock Docker install — server + 7 reference bots + your policy in slot 0,
producing `game started: players=8, imposters=2`. A **slot** is one of the 8 player seats; we
write a *minimal* `config.json` inline (pairing slot *i* with token `0xBADA55_i`) rather than
fetching the repo's, because the published image lags `master` and rejects `master`'s richer
config — a bare config with no `slots` block lets the engine assign roles and stays
image-version-agnostic:

```bash
# 0. one-time: a container network
docker network create crewrift-local || true

# ...and a minimal config (NO "slots" block -- the engine auto-assigns roles/colors).
# Fetching master's config.json instead will crash an older image on "slots[..].color".
cat > config.json <<'JSON'
{
  "tokens": ["0xBADA55_0","0xBADA55_1","0xBADA55_2","0xBADA55_3",
             "0xBADA55_4","0xBADA55_5","0xBADA55_6","0xBADA55_7"],
  "players": [{"name":"You"},{"name":"Bot1"},{"name":"Bot2"},{"name":"Bot3"},
              {"name":"Bot4"},{"name":"Bot5"},{"name":"Bot6"},{"name":"Bot7"}],
  "seed": 679961, "minPlayers": 8, "imposterCount": 2, "maxTicks": 10000,
  "mapPath": "data/croatoan.resources"
}
JSON

# 1. the game server (configured via env + the config mount; default CMD is /bin/crewrift)
docker run --rm -d --name crewrift-server --network crewrift-local -p 2000:2000 \
  -v "$PWD/config.json:/workspace/crewrift/config.json:ro" \
  -e COGAME_HOST=0.0.0.0 -e COGAME_PORT=2000 \
  -e COGAME_CONFIG_URI=file:///workspace/crewrift/config.json \
  public.ecr.aws/s3j4p9s7/treeform/games/crewrift:latest

# 2. wait for the server to be healthy before filling seats (avoids a connect race)
until curl -fsS http://localhost:2000/healthz >/dev/null 2>&1; do sleep 0.3; done

# 3. fill seats 1-7 with the reference bot; slot 0 stays open for you
for i in 1 2 3 4 5 6 7; do
  docker run --rm -d --name "crewrift-bot-$i" --network crewrift-local \
    -e COWORLD_PLAYER_WS_URL="ws://crewrift-server:2000/player?slot=$i&token=0xBADA55_$i" \
    public.ecr.aws/s3j4p9s7/treeform/players/notsus:latest
done

# 4. your policy takes slot 0 (the module §3i wrote):
export COWORLD_PLAYER_WS_URL="ws://localhost:2000/player?slot=0&token=0xBADA55_0"
cd my_starter_policy && python -m my_starter_policy

# 5. watch it live in a browser:  http://localhost:2000/client/global

# teardown:
docker rm -f crewrift-server
for i in 1 2 3 4 5 6 7; do docker rm -f "crewrift-bot-$i"; done
docker network rm crewrift-local
```

The next section packages exactly that runnable `my_starter_policy` module and submits it.


## 6 · Authenticate and submit your policy

Submission has two actors: **`softmax`** handles *auth*, and **`coworld`** does the *upload + submit*
(it uses your softmax login under the hood). These need a real account, Docker, and league access,
so the commands below are copy-paste for your terminal, and nothing is executed here.

Prerequisites: [`uv`](https://docs.astral.sh/uv/) (used throughout), `softmax`
(`uv tool install softmax-cli`), and `coworld` — clone
[Metta-AI/coworld](https://github.com/Metta-AI/coworld) and run it as `uv run coworld`
from that checkout (the game repo's play guide uses the same form).

### Package the policy

§3i already wrote everything this needs — a buildable project at `my_starter_policy/`
(`pyproject.toml`, the package, a `__main__` that calls `run_player(StarterPolicy())`).
`load_default_mesh()` ships the nav-mesh *inside* the `swgy_base` wheel, so there are no
assets to copy. To ship `DetectivePolicy` instead, swap the class name in the generated
`__main__.py`; both classes live in the same `policy.py`.

```dockerfile
# Dockerfile: vendor the wheels, install, run the module.
FROM docker.io/library/python:3.12-slim
WORKDIR /app
RUN pip install --no-cache-dir numpy websockets
COPY vendor/ /app/vendor/                         # swgy_base + my_starter_policy wheels
RUN pip install --no-cache-dir --no-deps /app/vendor/*.whl
CMD ["python", "-m", "my_starter_policy"]
```

```bash
# Build your policy wheel, stage its one dependency, then the image:
mkdir -p vendor
cp libs/swgy_base-*.whl vendor/
(cd my_starter_policy && uv build --wheel -o ../vendor/)   # pyproject written by §3i
docker build -t my-starter-policy:v1 .
```

### Authenticate (softmax-cli)

Submissions ride on a free **softmax** account: sign up (or sign in) at
[softmax.com](https://softmax.com). The `softmax login` below opens the same browser flow
and stores the credential locally.

```bash
uv tool install softmax-cli       # installs the `softmax` command
softmax login                     # opens a browser to log in
softmax status                    # confirm you're authenticated
```

### Submit (coworld)

```bash
coworld upload-policy my-starter-policy:v1 --name my-starter-policy
coworld submit       my-starter-policy:v1 --league league_605ff338-0a2e-4e62-aeda-559df9a9198f
```

That league id is the CrewRift league. Submitted? Two *optional* tooling tours follow —
rebuilding the nav-mesh (§7) and knob tuning (§8) — or jump straight to §9 for where the
loop goes next.

## 7 · (Optional) Rebuild the nav-mesh — for when movement is your bottleneck

Your policy is in the league; while it plays its first ranked games, open the hood on
the machinery you'd tune. (Spoiler from the analysis notebook: movement probably
*isn't* your bottleneck — league policies are already near-optimal at it. Skim this
until the data says otherwise.)

The nav-mesh is what `find_path` routes over. It's built in two layers: a **walkability grid**
(which pixels you can stand on) and a **waypoint graph** (a lattice of nodes + edges over the
walkable area). The canonical Croatoan mesh ships inside `swgy_base`, and `swgy_tools` lets us
*rebuild* the graph at any density straight from the walk grid, with no game assets needed.

We pull the walk mask out of the bundled mesh and rebuild the graph at a few **grid spacings**.
Denser spacing → more nodes → smoother paths but a bigger, slower graph.

In [ ]:
# (uses `mesh` from the §1 smoke test)
from swgy_tools.navmesh.build import build_graph, BuildParams

mask = mesh.grid.to_mask()          # (H, W) bool walkability, straight from the bundled mesh
print("walk mask:", mask.shape, "|", int(mask.sum()), "walkable px")

GRIDS = [8, 14, 22, 30]
built = {}
for g in GRIDS:
    nodes, edges = build_graph(mask, BuildParams(grid=g))
    built[g] = (nodes, edges)
    print(f"grid={g:2d}px -> {len(nodes):4d} nodes, {len(edges):4d} edges")

In [ ]:
from matplotlib.collections import LineCollection

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.patch.set_facecolor(PAGE)
for ax, g in zip(axes.ravel(), GRIDS):
    nodes, edges = built[g]
    ax.set_facecolor(PAGE)
    ax.imshow(mask, cmap=FLOOR_CMAP, extent=[0, mask.shape[1], mask.shape[0], 0],
              interpolation="nearest")
    xy = {n["id"]: (n["x"], n["y"]) for n in nodes}
    segs = [[xy[e["src"]], xy[e["dst"]]] for e in edges]      # one collection, not per-edge plots
    ax.add_collection(LineCollection(segs, colors="#3388ff", linewidths=0.35, zorder=1))
    xs = [n["x"] for n in nodes]; ys = [n["y"] for n in nodes]
    ax.scatter(xs, ys, s=3, c="#ff5533", zorder=2)
    ax.set_xlim(0, mask.shape[1]); ax.set_ylim(mask.shape[0], 0); ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(f"grid = {g}px  ({len(nodes)} nodes, {len(edges)} edges)",
                 color="white", fontsize=11)
fig.suptitle("Croatoan nav-mesh at four grid densities", fontsize=13, color="white")
plt.tight_layout(); plt.show()


### The density trade-off: planning CPU, not path length

Across sensible densities the *routes* come out near-identical — comparing their pixel length is a
red herring. What actually changes is **how long A\* takes to plan**: a denser mesh has more nodes
to expand per query. And planning CPU is exactly what your policy spends at runtime — it replans
each time its goal changes (a new task, a fresh victim, a drifted chase). On the deployed runtime
that is a **single, modest CPU core** (unlike the many-core box §8 uses for offline tuning), so the
density you ship is really a planning-latency choice. Let's measure that directly, at high
precision: wall-clock per `find_path`, many repeats, across each density.


In [ ]:
import time

from swgy_base.nav import NavMesh, NavNode, NavEdge, NavGrid

def to_mesh(mask, nodes, edges):
    g = NavGrid.from_mask(mask)
    N = [NavNode(id=n["id"], x=n["x"], y=n["y"], tags=frozenset(n["tags"])) for n in nodes]
    E = [NavEdge(src=e["src"], dst=e["dst"], weight=e["weight"],
                 directed=e["directed"], tags=frozenset(e["tags"])) for e in edges]
    return NavMesh(g, N, E)

# A spread of real routes (not one lucky pair) so the timing reflects typical planning.
PAIRS = [((181, 346), (971, 434)),   # Bridge -> Port Engine (the long haul)
         ((181, 346), (620, 300)),   # Bridge -> Storage Deck (mid-range)
         ((448, 154), (900, 320)),   # Observatory -> Reactor
         ((300, 520), (760, 190))]   # a cross-ship diagonal
REPS = 150                           # per pair, per density -- enough for a stable median

print(f"{'grid':>5} {'nodes':>7} {'edges':>7} {'plan median':>13} {'p95':>10}")
med_by_grid = {}
for g in GRIDS:
    m = to_mesh(mask, *built[g])
    for a, b in PAIRS:                       # warm caches / page-ins before timing
        find_path(m, a, b)
    samples = []
    for a, b in PAIRS:
        for _ in range(REPS):
            t0 = time.perf_counter()
            find_path(m, a, b)
            samples.append(time.perf_counter() - t0)
    samples.sort()
    med_by_grid[g] = samples[len(samples) // 2]
    p95 = samples[int(len(samples) * 0.95)]
    print(f"{g:5d} {len(m.nodes):7d} {len(m.edges):7d} "
          f"{med_by_grid[g] * 1e3:10.3f} ms {p95 * 1e3:7.3f} ms")

fine, coarse = med_by_grid[min(GRIDS)], med_by_grid[max(GRIDS)]
print(f"\nThe {min(GRIDS)}px mesh plans ~{fine / coarse:.0f}x slower than the {max(GRIDS)}px mesh "
      f"for near-identical routes.\nOn a one-core runtime that ratio -- not path length -- is the "
      f"whole trade-off.")


> **Building the mesh from raw game art.** We rebuilt only the *graph* from an existing walk grid.
> The full offline pipeline that produced the shipped asset, for anyone with the game's data
> (`$CREWRIFT_DATA_DIR`), is three `swgy_tools` / `swgy_base` commands:
>
> ```bash
> swgy-navmesh       --grid 14 --out tmp/croatoan.navmesh.json         # 1. geometry from the walk layer
> swgy-navmesh-sight --in tmp/croatoan.navmesh.json --out tmp/croatoan.vis.json --workers 8   # 2. bake visibility
> python -m swgy_base.nav compile tmp/croatoan.vis.json assets/croatoan.walk assets/croatoan.graph  # 3. compile
> ```
>
> Step 2 bakes per-node **exposure / witness** counts (how open each spot is), the data behind
> `mesh.has_visibility`, `mesh.exposure_at(x, y)`, and `mesh.nodes_by_exposure(...)`. An imposter
> policy can use it to find quiet corners, and a crew policy can use it to stay in the open.
> There are also `swgy-navmesh-viz` (render a mesh over the map) and `swgy-nav-bench`
> (an engine-faithful navigation benchmark).

## 8 · (Optional) Prove a change helps before spending a league submission on it

`NavParams` exposes 11 numeric knobs, each with documented genetic-algorithm (GA)
bounds in its `nav/params.py` docstring. `swgy_tune` turns a set of bounded knobs into a genome vector a search algorithm can
mutate, and `swgy_tools.navbench` supplies the fitness function: `enginesim`, a byte-level port
of the engine's player physics from `sim.nim`, plus a committed fixture of one hundred 8-task
tours. Everything below runs offline.

We wire the smallest honest version of that loop: expose a few knobs, drive one simulated leg,
then random-search a single tour. The real CLI, `swgy-nav-bench`, runs the full fixture with
paired statistics (bootstrap confidence intervals over 100 tours); treat this as the toy that
teaches the shape.

In [ ]:
from swgy_tune import Knob, KnobSpec
from swgy_base.nav import NavParams

# Three NavParams knobs with their documented GA bounds (see the nav/params.py docstrings).
SPEC = KnobSpec(knobs=[
    Knob("heuristic_weight", 1.0, 4.0, 1.0),    # A*: 1 = optimal, higher = greedier
    Knob("arrival_radius",   2.0, 48.0, 12.0),  # px within which a waypoint counts reached
    Knob("axis_deadband",    1.0, 12.0, 5.0),   # px of minor-axis error to ignore (anti-jitter)
])

genome = SPEC.default_vector()
print("knobs:", SPEC.names())
print("genome:", genome)
params = NavParams(**SPEC.from_vector(genome))
print("round-trip into NavParams:",
      (params.heuristic_weight, params.arrival_radius, params.axis_deadband))
# The genome is positional: keep the Knob order fixed for the life of a tuning run,
# or two runs will silently disagree about which number means what.

In [ ]:
from swgy_tools.navbench.enginesim import Engine, Player, TERMINAL_PX

def travel_ticks(params, start, goal_node, eng=None, player=None, max_ticks=4000):
    """Engine ticks for the follower to bring the collision point within 12px of a node.

    Runs the real loop offline: plan with A*, steer with NavState, integrate with the
    byte-faithful engine physics. Returns None if the leg does not complete in time.
    """
    eng = eng or Engine(mesh.grid)
    p = player or Player(x=start[0], y=start[1])
    gx, gy = mesh.node(goal_node).x, mesh.node(goal_node).y
    plan = find_path(mesh, (p.x, p.y), goal_node, params)
    if plan is None or len(plan) == 0:
        return None
    follower = NavState(plan=plan, params=params, grid=mesh.grid)
    for tick in range(max_ticks):
        if (p.x - gx) ** 2 + (p.y - gy) ** 2 <= 12.0 ** 2:
            return tick
        follower.update((p.x, p.y))
        dx, dy = follower.heading((p.x, p.y), collision=(p.x, p.y))
        eng.step(p, dx, dy)
    return None

reactor_station = mesh.task_node(next(t for t in mesh.tasks if t.room == "Reactor"))
ticks = travel_ticks(NavParams(), (181, 346), reactor_station)
print(f"Bridge -> Reactor station: {ticks} engine ticks ({ticks / 24:.1f}s at 24 ticks/s; "
      f"terminal speed {TERMINAL_PX:.2f}px/tick per axis)")

In [ ]:
import random
from swgy_tools.navbench.bench import load_sets

# One fixture tour: 8 task stations, visited in order, one persistent engine/player.
stations = [mesh.task_node(t) for t in mesh.tasks]
tour = [stations[i] for i in load_sets()[0]]
SPAWN = mesh.room_centroid("Bridge")

def tour_ticks(params):
    eng, p = Engine(mesh.grid), Player(x=SPAWN[0], y=SPAWN[1])
    total = 0
    for goal in tour:
        t = travel_ticks(params, (p.x, p.y), goal, eng=eng, player=p)
        if t is None:
            return None    # DNF: e.g. an arrival_radius above the 12px reach test parks short of it
        total += t
    return total

rng = random.Random(7)
baseline = tour_ticks(NavParams())
print(f"default NavParams : {baseline} ticks for the 8-task tour")
results = [("default", baseline)]
best_label, best_ticks = "default", baseline
for i in range(5):
    knobs = SPEC.from_vector(SPEC.clamp(SPEC.sample(rng)))
    t = tour_ticks(NavParams(**knobs))
    label = ", ".join(f"{k}={v:.1f}" for k, v in knobs.items())
    print(f"  sample {i}       : {t if t is not None else 'DNF':>5}  ({label})")
    results.append((f"sample {i}", t))
    if t is not None and t < best_ticks:
        best_label, best_ticks = label, t
print(f"best of this run  : {best_ticks} ticks ({best_label})")
print("swgy-nav-bench runs this over all 100 tours with paired stats; "
      "a knob only counts as a win when the confidence interval clears zero.")

# The run at a glance: green = best finisher, hatched = did not finish.
fig, ax = plt.subplots(figsize=(8, 2.6))
fig.patch.set_facecolor(PAGE); ax.set_facecolor(PAGE)
finished = [t for _, t in results if t is not None]
top = max(finished) * 1.15
for i, (lab, t) in enumerate(results):
    if t is None:
        ax.barh(i, top, color="#3a2430", edgecolor="#7a3550", hatch="//", height=0.62)
        ax.text(top * 0.985, i, "DNF", color="#ff7799", va="center", ha="right",
                fontsize=9, weight="bold")
    else:
        ax.barh(i, t, color="#22dd66" if t == min(finished) else "#1d7fd1", height=0.62)
        ax.text(t + top * 0.01, i, str(t), color="white", va="center", fontsize=9)
ax.set_yticks(range(len(results)), [lab for lab, _ in results], color="white", fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("engine ticks for the 8-task tour (lower is better)", color="white", fontsize=9)
ax.tick_params(colors="white")
for s in ax.spines.values():
    s.set_color("#3a4660")
plt.tight_layout(); plt.show()

## 9 · Iterate

You now hold the whole build half of the loop:

**build** a policy on `swgy_base` (§3) → **validate** it offline (§4) → **submit** it with
`softmax` + `coworld` (§6) → **analyze** its replays in **`LetsAnalyzeCrewrift.ipynb`** →
change **one** thing → **tune** that thing's knob against the offline benchmark (§8) →
resubmit and measure again.

Where to take the starter next:

- **Extend the brain.** `DetectivePolicy` shows the pattern: subclass, override one phase
  handler, inherit the rest. Prompt to try: *"add evidence-plus-tally voting: prefer a colour
  I personally saw vent, else back a clear tally leader, else skip — keep §2b's cursor rules."*
- **Let the data pick your battles.** The analysis notebook's scenarios are ordered by
  leverage: voting participation (the league literally gates on it), staying alive, and — as
  imposter — kill-site choice. Its final table lists the artifact to hand your coding agent
  for each.
- **Tune what you changed.** Any numeric constant you add is a knob. Prompt to try: *"wrap
  my new constant in a `Knob` with sane bounds and rerun the §8 loop; report the paired
  before/after."* The offline benchmark separates real wins from noise before you spend a
  league submission on them.

Optimise for the reward: **win +100, kill +10, task +1, missed vote −10, idling −1/20s** — the
win dwarfs everything, which is exactly what the analysis notebook's scoreboard shows.
